# KA SS 26 · Challenge 2 — **Stage 15 FINAL**
## Object-centric zoom-in ConvNeXt U-Net + continuous distance field + distance-watershed

**Single model. No ensemble. Final submission.**

This notebook compresses the strongest lessons from Stages 5–14 into one final pipeline.

### The key change

The reported leading approach detects cells, crops around each cell, and upscales those crops. The useful principle is not the detector itself; it is **object-centric effective resolution**. Stage 15 approximates that within the remaining T4 budget:

- **70% of training samples are genuinely cell-centred.** A labelled training instance is sampled uniformly from the global instance pool, a jittered native-resolution context window is cropped around it, and that patch is upscaled into a fixed 400×512 network crop.
- **30% remain random zoom-in context crops** so the model does not learn a trivial centred-cell prior.
- Uniform sampling over the global instance pool naturally exposes the model to dense images more often, because dense images contain more cells.
- The zoom factor is ~1.15–1.90×, centred near 1.5×.

### Target / decoder reformulation

The U-Net predicts **foreground + e70 seed + continuous normalised per-instance distance**. The binary seed is derived **after resizing** from the continuous distance field, avoiding the tiny-seed-resampling failure found in Stage 11. Final decoding probes seed/dist markers against foreground/distance watershed landscapes and falls back to the classic Stage-10 style family if distance decoding does not earn the win.

### Extra stability

- EMA model weights for checkpoint selection.
- exact 338/86 / 421-group validation assertion.
- official greedy Dice>=0.50 instance F1.
- RLE round-trip parity check.
- hard training-time guard + OOM-safe tiled inference.
- best checkpoint and probability maps mirrored to Drive.

### Run
1. Colab -> **T4 GPU**.
2. Upload `kaggle.json` to **`/content/kaggle.json`**.
3. **Run All.**


In [1]:
# ── 1 · Environment ───────────────────────────────────────────────────────────
import os, sys, gc, json, math, time, random, shutil, subprocess, warnings, itertools, zipfile, copy
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from PIL import Image

import scipy.ndimage as ndi
from scipy.ndimage import distance_transform_edt, binary_dilation
from scipy.sparse import csr_matrix
from skimage.segmentation import watershed

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

try:
    import timm
except ImportError:
    pip_install("timm")
    import timm

try:
    import cv2
    HAVE_CV2 = True
except ImportError:
    HAVE_CV2 = False

if shutil.which("kaggle") is None:
    pip_install("kaggle")

try:
    from torch.amp import autocast as _autocast, GradScaler as _GradScaler
    _AMP_NEW = True
except ImportError:
    from torch.cuda.amp import autocast as _autocast, GradScaler as _GradScaler
    _AMP_NEW = False

def amp_autocast():
    return _autocast("cuda", dtype=torch.float16) if _AMP_NEW else _autocast()

def make_scaler():
    return _GradScaler("cuda") if _AMP_NEW else _GradScaler()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert DEVICE.type == "cuda", "No GPU. Runtime -> Change runtime type -> T4 GPU."
torch.backends.cudnn.benchmark = True

T_START = time.time()
def elapsed():
    return time.time() - T_START

def fmt(sec):
    sec = int(sec)
    return f"{sec // 3600}h {(sec % 3600) // 60:02d}m {sec % 60:02d}s"

print("torch", torch.__version__, "| timm", timm.__version__, "| cv2", HAVE_CV2)
print("gpu:", torch.cuda.get_device_properties(0).name,
      f"| {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

torch 2.11.0+cu128 | timm 1.0.28 | cv2 True
gpu: Tesla T4 | 15.6 GB


In [2]:
# ── 2 · Configuration ─────────────────────────────────────────────────────────
class CFG:
    RUN_TAG = "stage15_final_objectcentric_zoomdist"
    SMOKE_TEST = False          # True -> ~8 min rehearsal, results meaningless
    RESUME = True               # reuse cached targets / checkpoint / maps if the cell reruns

    # ── wall clock ────────────────────────────────────────────────────────────
    TRAIN_DEADLINE_SEC = 60 * 60      # training stops here no matter what
    TOTAL_BUDGET_SEC   = 105 * 60     # everything else must fit under this

    IMG_H, IMG_W = 520, 704
    PAD_MULT = 32

    # ── split: identical construction to Stages 7-12, asserted ────────────────
    SPLIT_SEED = 2026
    DUP_CORR_THR = 0.72
    N_FOLDS = 5
    FOLD = 0
    EXPECT_TRAIN, EXPECT_VAL, EXPECT_GROUPS = 338, 86, 421
    STRICT_SPLIT = True

    ENCODER = "convnext_tiny.fb_in22k_ft_in1k"
    ENCODER_FALLBACKS = ("convnext_tiny",)   # a silent resnet fallback would void the run
    DECODER_CHANNELS = (256, 128, 64, 32)
    OUT_CHANNELS = 3                          # fg, seed, ndist

    # ── targets ───────────────────────────────────────────────────────────────
    SEED_EROSION_FRAC = 0.70   # Stage 10's measured optimum, now only the TRAIN-time seed
    BORDER_DILATION = 2
    BORDER_WEIGHT = 4.0
    DIST_WEIGHT = 1.5          # weight of the distance-regression term in the loss
    DIST_FG_WEIGHT = 3.0
    DIST_BG_WEIGHT = 0.10       # extra weight on foreground pixels inside that term

    # ── schedule: Stage 5's, with the crop made encoder-friendly ──────────────
    EPOCHS = 72
    STEPS_PER_EPOCH = 42
    BATCH_SIZE = 8
    CROP = (400, 512)
    LR = 3e-4
    ENCODER_LR_MULT = 0.25
    WD = 1e-4
    FREEZE_ENCODER_EPOCHS = 2
    WARMUP_STEPS = 150
    GRAD_CLIP = 5.0
    NUM_WORKERS = 2
    EVAL_START = 24
    EVAL_EVERY = 4
    QUICK_VAL_N = 32
    QUICK_SCALE = 1.5

    # ── augmentation: ZOOM IN. mixture is (probability, lo, hi) ───────────────
    SCALE_MIX = ((0.25, 1.15, 1.35),
                 (0.50, 1.35, 1.65),
                 (0.25, 1.65, 1.90))
    OBJECT_CENTRIC_P = 0.70
    CENTER_JITTER_FRAC = 0.18
    AUG_D4 = True              # 90-degree rotations; phase contrast has no canonical up
    AUG_HFLIP = 0.5
    AUG_VFLIP = 0.5
    AUG_BRIGHT = 0.15          # in units of dataset sigma, Stage 5 values
    AUG_CONTRAST = 0.15
    AUG_NOISE = 0.02
    AUG_BLUR_P = 0.15          # upscaled crops are smooth; jitter the sharpness a little
    AUG_GAMMA = 0.12

    # ── inference ─────────────────────────────────────────────────────────────
    TTA_SCALES = (1.25, 1.50, 1.75)
    TTA_FLIPS = True
    TILE_PIXEL_LIMIT = 1_400_000   # above this a view is tiled instead of run whole
    TILE = 768
    TILE_OVERLAP = 128

    # ── decode ────────────────────────────────────────────────────────────────
    DEFAULT_POST = dict(marker="seed", landscape="dist",
                        t_cell=0.50, t_marker=0.65, min_area=90)
    COARSE_FAMILIES = (("seed", "dist"), ("seed", "fg"), ("dist", "dist"), ("dist", "fg"))
    COARSE_N = 40
    T_CELL_GRID = (0.40, 0.45, 0.50, 0.55, 0.60)
    T_MARKER_GRID = (0.50, 0.55, 0.60, 0.65, 0.70, 0.75)
    MIN_AREA_GRID = (50, 70, 90, 110)
    PLATEAU_TOL = 0.004

    # ── density-conditioned t_cell (Stage 9 measured +0.0049 for this idea) ───
    DENSITY_EDGES = (0, 50, 200, 400, 10 ** 9)
    DENSITY_LABELS = ("sparse <50", "medium 50-199", "dense 200-399", "very dense >=400")
    ROUTE_MIN_GAIN = 0.004      # must beat the global decode by this on the 86 val images
    ROUTE_MIN_WINRATE = 0.90    # and win this often under an image-level bootstrap

    EMA_DECAY = 0.9985

    SUB_INDEX_COL = "idx"

    # ── reference numbers to beat (public LB in brackets) ────────────────────
    REF_STAGE10_LOCAL = 0.8072   # (0.5295)
    REF_STAGE12_LOCAL = 0.8119   # (0.5335)  <- the bar


if CFG.SMOKE_TEST:
    CFG.EPOCHS = 4
    CFG.EVAL_START = 2
    CFG.EVAL_EVERY = 2
    CFG.QUICK_VAL_N = 8
    CFG.TTA_SCALES = (1.5,)
    CFG.TTA_FLIPS = False
    CFG.COARSE_N = 8
    CFG.T_CELL_GRID = (0.45, 0.50)
    CFG.T_MARKER_GRID = (0.60, 0.65)
    CFG.MIN_AREA_GRID = (70, 90)
    CFG.TRAIN_DEADLINE_SEC = 12 * 60
    print(">>> SMOKE TEST — this only proves the pipeline runs. Numbers are meaningless.")


def seed_everything(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

seed_everything(CFG.SPLIT_SEED)
STATS = {"run_tag": CFG.RUN_TAG, "smoke_test": CFG.SMOKE_TEST}
print("run tag:", CFG.RUN_TAG)
print("scale mixture:", CFG.SCALE_MIX, "| crop", CFG.CROP, "| TTA", CFG.TTA_SCALES)

run tag: stage15_final_objectcentric_zoomdist
scale mixture: ((0.25, 1.15, 1.35), (0.5, 1.35, 1.65), (0.25, 1.65, 1.9)) | crop (400, 512) | TTA (1.25, 1.5, 1.75)


In [3]:
# ── 3 · Kaggle data, working directories, encoder preflight ───────────────────
KAGGLE_COMPETITION = "ka-ss-26-challenge-2"
DATA_ROOT = Path("/content/data")
WORK = Path("/content/stage15")
CKPT, ART, CACHE, MAPS = WORK / "ckpt", WORK / "artifacts", WORK / "cache", WORK / "maps"
for p in (DATA_ROOT, WORK, CKPT, ART, CACHE, MAPS):
    p.mkdir(parents=True, exist_ok=True)

# Drive is optional. If it mounts, artifacts are mirrored there; if not, the run continues.
DRIVE_DIR = None
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_DIR = Path("/content/drive/MyDrive/ka_ss26/stage15")
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    print("drive mirror:", DRIVE_DIR)
except Exception as exc:
    print("no Drive mirror (continuing):", type(exc).__name__)

KAGGLE_JSON = None
for cand in (Path("/content/kaggle.json"),
             Path("/content/drive/MyDrive/ka_ss26/kaggle.json"),
             Path.home() / ".kaggle/kaggle.json"):
    if cand.is_file():
        KAGGLE_JSON = cand
        break
assert KAGGLE_JSON is not None, "Upload kaggle.json to /content, then rerun this cell."
KDIR = Path.home() / ".kaggle"
KDIR.mkdir(parents=True, exist_ok=True)
if KAGGLE_JSON.resolve() != (KDIR / "kaggle.json").resolve():
    shutil.copy2(KAGGLE_JSON, KDIR / "kaggle.json")
(KDIR / "kaggle.json").chmod(0o600)
print("kaggle credentials:", KAGGLE_JSON)


def looks_like_dataset(p):
    return (p / "train.csv").is_file() and (p / "train").is_dir() and (p / "test").is_dir()

def find_dataset_root(base):
    if looks_like_dataset(base):
        return base
    for csv_path in base.rglob("train.csv"):
        if looks_like_dataset(csv_path.parent):
            return csv_path.parent
    return None

BASE = find_dataset_root(DATA_ROOT)
if BASE is None:
    print(">>> downloading competition data ...")
    r = subprocess.run(["kaggle", "competitions", "download", "-c", KAGGLE_COMPETITION,
                        "-p", str(DATA_ROOT), "--force"], capture_output=True, text=True)
    print(r.stdout[-800:])
    if r.returncode != 0:
        print(r.stderr[-800:])
        raise RuntimeError("Kaggle download failed. Joined the competition? Rules accepted?")
    for z in sorted(DATA_ROOT.glob("*.zip")):
        print("extracting", z.name)
        with zipfile.ZipFile(z) as zf:
            zf.extractall(DATA_ROOT)
    BASE = find_dataset_root(DATA_ROOT)

assert BASE is not None, "Downloaded, but train/test structure not found."
TRAIN_DIR, TEST_DIR, TRAIN_CSV = BASE / "train", BASE / "test", BASE / "train.csv"
print("data root:", BASE)


def resolve_encoder():
    for name in (CFG.ENCODER,) + tuple(CFG.ENCODER_FALLBACKS):
        try:
            m = timm.create_model(name, pretrained=True, features_only=True, in_chans=3)
            ch = m.feature_info.channels()
            del m; gc.collect()
            print(f"encoder ok: {name} | channels {ch}")
            return name
        except Exception as exc:
            print(f"  [{name}] unavailable: {type(exc).__name__}: {str(exc)[:140]}")
    raise RuntimeError("No pretrained encoder available.")

CFG.ENCODER = resolve_encoder()
assert CFG.ENCODER.startswith("convnext_tiny"), "Wrong encoder family — stop and fix."
STATS["encoder"] = CFG.ENCODER

Mounted at /content/drive
drive mirror: /content/drive/MyDrive/ka_ss26/stage15
kaggle credentials: /content/kaggle.json
>>> downloading competition data ...


extracting ka-ss-26-challenge-2.zip
data root: /content/data/cells_segmentation


model.safetensors: reconstructing file:   0%|          |  0.00B /  114MB            

model.safetensors: downloading bytes:           |  0.00B            

encoder ok: convnext_tiny.fb_in22k_ft_in1k | channels [96, 192, 384, 768]


In [4]:
# ── 4 · Labels, RLE codec, official metric ────────────────────────────────────
df_raw = pd.read_csv(TRAIN_CSV)
ID_COL = next(c for c in df_raw.columns if c.lower() in ("id", "image_id", "imageid"))
ANN_COL = next(c for c in df_raw.columns
               if c.lower() in ("annotation", "encodedpixels", "rle", "segmentation"))

TRAIN_IDS = sorted(p.stem for p in TRAIN_DIR.glob("*.png"))
TEST_IDS = sorted(p.stem for p in TEST_DIR.glob("*.png"))
with Image.open(TRAIN_DIR / f"{TRAIN_IDS[0]}.png") as im:
    CFG.IMG_W, CFG.IMG_H = im.size
PAD_H = int(math.ceil(CFG.IMG_H / CFG.PAD_MULT) * CFG.PAD_MULT)
PAD_W = int(math.ceil(CFG.IMG_W / CFG.PAD_MULT) * CFG.PAD_MULT)
NPIX = CFG.IMG_H * CFG.IMG_W
assert len(TRAIN_IDS) == 424 and len(TEST_IDS) == 182, (len(TRAIN_IDS), len(TEST_IDS))
print(f"train {len(TRAIN_IDS)} | test {len(TEST_IDS)} | {CFG.IMG_H}x{CFG.IMG_W}")


def rle_decode_flat(rle, h, w, order="C"):
    if not isinstance(rle, str) or not rle.strip() or rle.strip() == "-1":
        return np.empty(0, dtype=np.int64)
    values = np.asarray(rle.split(), dtype=np.int64)
    starts, lengths = values[0::2] - 1, values[1::2]
    idx = np.concatenate([np.arange(s, s + l) for s, l in zip(starts, lengths)])
    return idx if order == "C" else (idx % h) * w + (idx // h)


def rle_encode(mask_flat_u8):
    pixels = np.concatenate([[0], mask_flat_u8, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[0::2]
    return " ".join(str(int(x)) for x in runs)


def flat_to_rle(flat):
    # flat is C-order pixel indices; the string is emitted in the detected order.
    mask = np.zeros(NPIX, dtype=np.uint8)
    mask[np.asarray(flat, dtype=np.int64)] = 1
    if RLE_ORDER == "F":
        mask = np.asfortranarray(mask.reshape(CFG.IMG_H, CFG.IMG_W)).reshape(-1, order="F")
    return rle_encode(mask)


probe_df = df_raw[df_raw[ANN_COL].astype(str) != "-1"].sample(min(200, len(df_raw)), random_state=0)
order_report = {}
for order in ("C", "F"):
    single = []
    for rle in probe_df[ANN_COL].astype(str):
        flat = rle_decode_flat(rle, CFG.IMG_H, CFG.IMG_W, order)
        if flat.size == 0 or flat.max() >= NPIX:
            continue
        m = np.zeros(NPIX, bool); m[flat] = True
        single.append(ndi.label(m.reshape(CFG.IMG_H, CFG.IMG_W))[1] == 1)
    order_report[order] = float(np.mean(single) * 100) if single else 0.0
RLE_ORDER = max(order_report, key=lambda o: order_report[o])
print("RLE_ORDER =", RLE_ORDER, {k: round(v, 1) for k, v in order_report.items()})
assert order_report[RLE_ORDER] > 90

INST = defaultdict(list)
for image_id, rle in zip(df_raw[ID_COL].astype(str), df_raw[ANN_COL].astype(str)):
    flat = rle_decode_flat(rle, CFG.IMG_H, CFG.IMG_W, RLE_ORDER)
    if flat.size:
        INST[image_id].append(np.unique(flat).astype(np.int64))

bad = 0
for image_id in TRAIN_IDS[:40]:
    for flat in INST[image_id]:
        rt = np.unique(rle_decode_flat(flat_to_rle(flat), CFG.IMG_H, CFG.IMG_W, RLE_ORDER))
        if not np.array_equal(rt, flat):
            bad += 1
assert bad == 0, f"RLE round-trip lossy on {bad} instances."
print(">>> RLE round-trip exact on 40 images.")

counts = np.array([len(INST[i]) for i in TRAIN_IDS])
areas = np.array([len(f) for i in TRAIN_IDS for f in INST[i]])
STATS["gt_mean_instances"] = float(counts.mean())
print(f"instances/image mean {counts.mean():.1f} max {counts.max()} | area p1 "
      f"{int(np.percentile(areas, 1))} p50 {int(np.percentile(areas, 50))}")


def sparse_masks(flats):
    if not flats:
        return csr_matrix((0, NPIX), dtype=np.float32)
    indptr = np.zeros(len(flats) + 1, np.int64)
    indptr[1:] = np.cumsum([len(f) for f in flats])
    indices = np.concatenate(flats).astype(np.int64)
    return csr_matrix((np.ones(indices.size, np.float32), indices, indptr),
                      shape=(len(flats), NPIX))


def dice_matrix(gt, pred):
    if len(gt) == 0 or len(pred) == 0:
        return np.zeros((len(gt), len(pred)), np.float32)
    a, b = sparse_masks(gt), sparse_masks(pred)
    inter = (a @ b.T).toarray()
    aa = np.asarray(a.sum(axis=1)).reshape(-1, 1)
    bb = np.asarray(b.sum(axis=1)).reshape(1, -1)
    return (2.0 * inter / np.maximum(aa + bb, 1.0)).astype(np.float32)


def instance_f1(gt, pred, thr=0.50):
    # Official metric: greedy one-to-one matching, a match needs Dice >= 0.50.
    ng, npred = len(gt), len(pred)
    if ng == 0 and npred == 0:
        return 1.0
    if ng == 0 or npred == 0:
        return 0.0
    d = dice_matrix(gt, pred)
    gi, pi = np.nonzero(d >= thr)
    if gi.size == 0:
        return 0.0
    order = np.argsort(-d[gi, pi])
    ug, up, tp = set(), set(), 0
    for g, p in zip(gi[order], pi[order]):
        g, p = int(g), int(p)
        if g in ug or p in up:
            continue
        ug.add(g); up.add(p); tp += 1
    return 2.0 * tp / (ng + npred)


_x = np.array([0, 1, 2], np.int64)
assert abs(instance_f1([_x], [_x]) - 1.0) < 1e-8
assert abs(instance_f1([_x], [_x, _x]) - 2.0 / 3.0) < 1e-8
assert instance_f1([_x], []) == 0.0
print(">>> official metric self-tests passed")

train 424 | test 182 | 520x704
RLE_ORDER = C {'C': 91.5, 'F': 0.0}
>>> RLE round-trip exact on 40 images.
instances/image mean 125.2 max 782 | area p1 53 p50 215
>>> official metric self-tests passed


In [5]:
# ── 5 · Images and the grouped split (must reproduce 338 / 86 / 421) ──────────
def load_gray_u8(path):
    with Image.open(path) as im:
        return np.asarray(im.convert("L"), dtype=np.uint8)

IMAGES = {i: load_gray_u8(TRAIN_DIR / f"{i}.png") for i in TRAIN_IDS}
MEAN = float(np.mean([IMAGES[i].mean() for i in TRAIN_IDS]) / 255.0)
STD = float(np.mean([IMAGES[i].std() for i in TRAIN_IDS]) / 255.0)
print(f"loaded {len(IMAGES)} images | mean/std {MEAN:.4f}/{STD:.4f}")

TH_H, TH_W = 65, 88

def thumbnail_vec(arr):
    a = np.asarray(Image.fromarray(arr).resize((TH_W, TH_H), Image.BILINEAR), np.float32).ravel()
    a -= a.mean()
    n = np.linalg.norm(a)
    return a / n if n > 0 else a

THUMBS = np.stack([thumbnail_vec(IMAGES[i]) for i in TRAIN_IDS])
CORR = THUMBS @ THUMBS.T
np.fill_diagonal(CORR, 0.0)

adj = CORR >= CFG.DUP_CORR_THR
group_of = -np.ones(len(TRAIN_IDS), int)
g = 0
for i in range(len(TRAIN_IDS)):
    if group_of[i] >= 0:
        continue
    stack, group_of[i] = [i], g
    while stack:
        u = stack.pop()
        for v in np.nonzero(adj[u])[0]:
            if group_of[v] < 0:
                group_of[v] = g
                stack.append(v)
    g += 1
n_groups = g

rng = np.random.RandomState(CFG.SPLIT_SEED)
order = rng.permutation(n_groups)
fold_of_group = {int(gid): int(k % CFG.N_FOLDS) for k, gid in enumerate(order)}
val_mask = np.array([fold_of_group[int(x)] == CFG.FOLD for x in group_of])
train_ids = [i for i, m in zip(TRAIN_IDS, val_mask) if not m]
val_ids = [i for i, m in zip(TRAIN_IDS, val_mask) if m]

vi, ti = np.nonzero(val_mask)[0], np.nonzero(~val_mask)[0]
max_corr = float(CORR[np.ix_(vi, ti)].max())
print(f"groups {n_groups} | train/val {len(train_ids)}/{len(val_ids)} | max val->train corr {max_corr:.3f}")
assert max_corr < CFG.DUP_CORR_THR, "Near-duplicate leaked across the split."

got = (len(train_ids), len(val_ids), n_groups)
want = (CFG.EXPECT_TRAIN, CFG.EXPECT_VAL, CFG.EXPECT_GROUPS)
if got != want:
    msg = f"SPLIT MISMATCH: got {got}, Stages 7-12 had {want}. Local F1 is not comparable."
    if CFG.STRICT_SPLIT:
        raise RuntimeError(msg)
    print("!!", msg)
print("val_ids[:5]:", val_ids[:5], "...", val_ids[-2:])

GT_COUNT = {i: len(INST[i]) for i in TRAIN_IDS}
STATS.update(n_groups=int(n_groups), max_val_train_corr=max_corr,
             n_train=len(train_ids), n_val=len(val_ids))
(ART / "val_ids.json").write_text(json.dumps(
    {"val_ids": val_ids, "n_train": len(train_ids), "n_groups": int(n_groups)}, indent=1))

# Global training-instance pool for the object-centric curriculum.
# Uniform instance sampling approximates the leader's "train around every cell" idea
# and naturally increases dense-scene exposure without an arbitrary repeat factor.
TRAIN_INSTANCE_POOL = []
for image_id in train_ids:
    for k, flat in enumerate(INST[image_id]):
        ys, xs = np.divmod(flat, CFG.IMG_W)
        TRAIN_INSTANCE_POOL.append(
            (image_id, k, float(ys.mean()), float(xs.mean()), int(flat.size))
        )

print("training instance pool:", len(TRAIN_INSTANCE_POOL))
assert len(TRAIN_INSTANCE_POOL) > 30000

# Stratified quick validation set: checkpoint selection must see sparse AND dense images.
def _gt_bin(n):
    return 0 if n < 50 else 1 if n < 200 else 2 if n < 400 else 3

_qbins = defaultdict(list)
for image_id in val_ids:
    _qbins[_gt_bin(len(INST[image_id]))].append(image_id)

QUICK_IDS = []
_quota = max(1, CFG.QUICK_VAL_N // 4)
for b in range(4):
    ids = sorted(_qbins[b])
    take = min(_quota, len(ids))
    if take:
        pos = np.linspace(0, len(ids) - 1, take).round().astype(int)
        QUICK_IDS.extend([ids[j] for j in pos])

for image_id in val_ids:
    if len(QUICK_IDS) >= CFG.QUICK_VAL_N:
        break
    if image_id not in QUICK_IDS:
        QUICK_IDS.append(image_id)
QUICK_IDS = QUICK_IDS[:CFG.QUICK_VAL_N]
assert len(QUICK_IDS) == CFG.QUICK_VAL_N
print("quick-val images:", len(QUICK_IDS), "| GT-count range",
      min(len(INST[i]) for i in QUICK_IDS), "..", max(len(INST[i]) for i in QUICK_IDS))


loaded 424 images | mean/std 0.5018/0.0522
groups 421 | train/val 338/86 | max val->train corr 0.665
val_ids[:5]: ['0323e81d23d9', '0e9b40b10de8', '115fad550598', '1242e4ec6ef3', '144a7a69f67d'] ... ['fd98d82784a1', 'fe33dfcf4ebd']
training instance pool: 42981
quick-val images: 32 | GT-count range 11 .. 609


In [6]:
# ── 6 · Targets: foreground, CONTINUOUS normalised distance, contact zone ─────
# ch0 foreground (0/1)                              -> mask channel
# ch1 ndist = per-instance EDT / EDT.max, 0..255     -> seeds are THRESHOLDED FROM THIS,
#                                                      after resizing, at any erosion level
# ch2 contact zone (0/1)                             -> loss weighting only
#
# Why this matters: Stage 11 resized a 1.56%-coverage binary seed core with nearest
# neighbour and silently deleted small cells' cores. A normalised distance field is
# scale-invariant and interpolates cleanly, so "erode by 0.70" survives any resize.
def build_targets(image_id):
    h, w = CFG.IMG_H, CFG.IMG_W
    fg = np.zeros((h, w), np.uint8)
    nd = np.zeros((h, w), np.float32)
    overlap = np.zeros((h, w), np.uint16)
    for flat in INST[image_id]:
        ys, xs = np.divmod(flat, w)
        y0, y1 = int(ys.min()), int(ys.max()) + 1
        x0, x1 = int(xs.min()), int(xs.max()) + 1
        sub = np.zeros((y1 - y0, x1 - x0), bool)
        sub[ys - y0, xs - x0] = True
        fg[y0:y1, x0:x1] |= sub
        dist = distance_transform_edt(np.pad(sub, 1))[1:-1, 1:-1]
        dmax = float(dist.max())
        if dmax > 0:
            region = nd[y0:y1, x0:x1]
            np.maximum(region, (dist / dmax).astype(np.float32), out=region)
        pad = CFG.BORDER_DILATION
        ys0, ys1 = max(0, y0 - pad), min(h, y1 + pad)
        xs0, xs1 = max(0, x0 - pad), min(w, x1 + pad)
        big = np.zeros((ys1 - ys0, xs1 - xs0), bool)
        big[y0 - ys0:y1 - ys0, x0 - xs0:x1 - xs0] = sub
        overlap[ys0:ys1, xs0:xs1] += binary_dilation(big, iterations=pad).astype(np.uint16)
    return np.stack([fg,
                     np.clip(nd * 255.0, 0, 255).astype(np.uint8),
                     (overlap >= 2).astype(np.uint8)], 0)


TARGET_CACHE = CACHE / "targets_ndist.npz"
if CFG.RESUME and TARGET_CACHE.exists():
    with np.load(TARGET_CACHE) as z:
        TARGETS = {k: z[k] for k in z.files}
    print("targets loaded from cache")
else:
    t0 = time.time()
    TARGETS = {i: build_targets(i) for i in TRAIN_IDS}
    np.savez_compressed(TARGET_CACHE, **TARGETS)
    print(f"targets built in {time.time() - t0:.0f}s")

SEED_U8 = int(round(CFG.SEED_EROSION_FRAC * 255))
cov_fg = float(np.mean([TARGETS[i][0].mean() for i in TRAIN_IDS]) * 100)
cov_seed = float(np.mean([(TARGETS[i][1] > SEED_U8).mean() for i in TRAIN_IDS]) * 100)
print(f"foreground {cov_fg:.2f}% | derived seed at erosion {CFG.SEED_EROSION_FRAC}: {cov_seed:.2f}%")
# Stage 10 measured 1.56% at erosion 0.70 — this should land within a hair of that.
assert 0.8 < cov_seed < 3.0, "Derived seed coverage is off; the distance field is wrong."

targets built in 25s
foreground 10.87% | derived seed at erosion 0.7: 1.55%


In [7]:
# ── 7 · Dataset: OBJECT-CENTRIC zoom-in + SHAPE-SAFE D4 ──────────────────────
# Important:
# CFG.CROP is rectangular (400, 512). A 90°/270° rotation of an already-final
# crop becomes (512, 400), which caused the previous sanity-check failure.
#
# Fix:
# choose the D4 rotation BEFORE deciding the pre-rotation crop dimensions.
# For odd rotations, sample a (512, 400)-shaped pre-rotation crop so that
# rotating it yields exactly (400, 512). No aspect-ratio distortion, and full
# 90° D4 augmentation is preserved.

def sample_scale():
    r, acc = random.random(), 0.0
    for p, lo, hi in CFG.SCALE_MIX:
        acc += p
        if r < acc:
            return random.uniform(lo, hi)
    _, lo, hi = CFG.SCALE_MIX[-1]
    return random.uniform(lo, hi)


def resize_to(img, tgt, out_h, out_w):
    if HAVE_CV2:
        img_r = cv2.resize(img, (out_w, out_h), interpolation=cv2.INTER_LINEAR)
        fg = cv2.resize(tgt[0], (out_w, out_h), interpolation=cv2.INTER_NEAREST)
        nd = cv2.resize(tgt[1], (out_w, out_h), interpolation=cv2.INTER_LINEAR)
        bd = cv2.resize(tgt[2], (out_w, out_h), interpolation=cv2.INTER_NEAREST)
    else:
        img_r = np.asarray(
            Image.fromarray(np.clip(img * 255, 0, 255).astype(np.uint8)).resize(
                (out_w, out_h), Image.BILINEAR
            ),
            np.float32,
        ) / 255.0
        fg = np.asarray(
            Image.fromarray(tgt[0]).resize((out_w, out_h), Image.NEAREST)
        )
        nd = np.asarray(
            Image.fromarray(tgt[1]).resize((out_w, out_h), Image.BILINEAR)
        )
        bd = np.asarray(
            Image.fromarray(tgt[2]).resize((out_w, out_h), Image.NEAREST)
        )

    return (
        img_r.astype(np.float32),
        np.stack([fg, nd, bd]).astype(np.uint8),
    )


def crop_native_with_pad(img, tgt, cy, cx, crop_h, crop_w):
    y0 = int(round(cy - crop_h / 2))
    x0 = int(round(cx - crop_w / 2))
    y1, x1 = y0 + crop_h, x0 + crop_w

    pt, pl = max(0, -y0), max(0, -x0)
    pb = max(0, y1 - img.shape[0])
    pr = max(0, x1 - img.shape[1])

    if pt or pl or pb or pr:
        img = np.pad(img, ((pt, pb), (pl, pr)), mode="reflect")
        tgt = np.pad(
            tgt,
            ((0, 0), (pt, pb), (pl, pr)),
            mode="constant",
        )
        y0 += pt
        x0 += pl

    return (
        img[y0:y0 + crop_h, x0:x0 + crop_w],
        tgt[:, y0:y0 + crop_h, x0:x0 + crop_w],
    )


def random_zoom_context(img, tgt, scale, out_h, out_w):
    nh = max(out_h, int(round(img.shape[0] * scale)))
    nw = max(out_w, int(round(img.shape[1] * scale)))

    img_r, tgt_r = resize_to(img, tgt, nh, nw)

    y0 = random.randint(0, nh - out_h)
    x0 = random.randint(0, nw - out_w)

    return (
        img_r[y0:y0 + out_h, x0:x0 + out_w],
        tgt_r[:, y0:y0 + out_h, x0:x0 + out_w],
    )


def pad_to(arr, height, width, mode="reflect"):
    ph = max(0, height - arr.shape[-2])
    pw = max(0, width - arr.shape[-1])

    if ph == 0 and pw == 0:
        return arr

    pad_width = [(0, 0)] * (arr.ndim - 2) + [(0, ph), (0, pw)]
    kw = {} if mode == "reflect" else {"constant_values": 0}

    return np.pad(arr, pad_width, mode=mode, **kw)


def sample_d4_k():
    return random.randint(0, 3) if CFG.AUG_D4 else 0


def apply_d4_with_k(img, tgt, k):
    if k:
        img = np.rot90(img, k)
        tgt = np.rot90(tgt, k, axes=(1, 2))

    if random.random() < CFG.AUG_HFLIP:
        img = img[:, ::-1]
        tgt = tgt[:, :, ::-1]

    if random.random() < CFG.AUG_VFLIP:
        img = img[::-1, :]
        tgt = tgt[:, ::-1, :]

    return (
        np.ascontiguousarray(img),
        np.ascontiguousarray(tgt),
    )


def photometric(img):
    img = img + random.uniform(
        -CFG.AUG_BRIGHT,
        CFG.AUG_BRIGHT,
    ) * STD

    f = 1.0 + random.uniform(
        -CFG.AUG_CONTRAST,
        CFG.AUG_CONTRAST,
    )
    img = (img - MEAN) * f + MEAN

    if CFG.AUG_GAMMA > 0:
        gamma = 1.0 + random.uniform(
            -CFG.AUG_GAMMA,
            CFG.AUG_GAMMA,
        )
        img = np.clip(img, 1e-4, 1.0) ** gamma

    if HAVE_CV2 and random.random() < CFG.AUG_BLUR_P:
        img = cv2.GaussianBlur(
            img,
            (3, 3),
            random.uniform(0.25, 0.85),
        )

    img = img + (
        np.random.randn(*img.shape).astype(np.float32)
        * CFG.AUG_NOISE
        * STD
    )

    return np.clip(img, 0.0, 1.0).astype(np.float32)


class CellDataset(Dataset):
    def __init__(self, ids, crop, training=True):
        self.ids = list(ids)
        self.crop_h, self.crop_w = crop
        self.training = bool(training)

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, index):
        if self.training:
            k = sample_d4_k()

            # Before an odd 90° rotation, swap the requested pre-rotation
            # dimensions. After rotation the patch is exactly crop_h x crop_w.
            if k % 2:
                pre_h = self.crop_w
                pre_w = self.crop_h
            else:
                pre_h = self.crop_h
                pre_w = self.crop_w
        else:
            k = 0
            pre_h = self.crop_h
            pre_w = self.crop_w

        if self.training and random.random() < CFG.OBJECT_CENTRIC_P:
            # True object-centred native crop -> rotate/flip -> isotropic upscale.
            image_id, _, cy, cx, _ = random.choice(TRAIN_INSTANCE_POOL)

            img = IMAGES[image_id].astype(np.float32) / 255.0
            tgt = TARGETS[image_id]

            scale = sample_scale()

            src_h = max(32, int(round(pre_h / scale)))
            src_w = max(32, int(round(pre_w / scale)))

            cy += random.uniform(
                -CFG.CENTER_JITTER_FRAC,
                CFG.CENTER_JITTER_FRAC,
            ) * src_h

            cx += random.uniform(
                -CFG.CENTER_JITTER_FRAC,
                CFG.CENTER_JITTER_FRAC,
            ) * src_w

            img, tgt = crop_native_with_pad(
                img,
                tgt,
                cy,
                cx,
                src_h,
                src_w,
            )

            img, tgt = apply_d4_with_k(
                img,
                tgt,
                k,
            )

            # Because pre_h/pre_w were swapped for odd k, this resize is
            # isotropic apart from integer rounding and always ends at 400x512.
            img, tgt = resize_to(
                img,
                tgt,
                self.crop_h,
                self.crop_w,
            )

        else:
            # Random context branch.
            image_id = self.ids[index % len(self.ids)]

            img = IMAGES[image_id].astype(np.float32) / 255.0
            tgt = TARGETS[image_id]

            img, tgt = random_zoom_context(
                img,
                tgt,
                sample_scale(),
                pre_h,
                pre_w,
            )

            if self.training:
                img, tgt = apply_d4_with_k(
                    img,
                    tgt,
                    k,
                )

                # Odd rotations turn pre_h x pre_w back into crop_h x crop_w.
                assert img.shape == (
                    self.crop_h,
                    self.crop_w,
                ), (
                    "D4 shape bug: "
                    f"got {img.shape}, expected "
                    f"{(self.crop_h, self.crop_w)}"
                )

        if self.training:
            img = photometric(img)
        else:
            img = pad_to(
                img,
                PAD_H,
                PAD_W,
            )
            tgt = pad_to(
                tgt,
                PAD_H,
                PAD_W,
                mode="constant",
            )

        # Final hard shape check BEFORE tensor creation.
        if self.training:
            assert img.shape == (
                self.crop_h,
                self.crop_w,
            ), (
                f"training image shape {img.shape} != "
                f"{(self.crop_h, self.crop_w)}"
            )

            assert tgt.shape == (
                3,
                self.crop_h,
                self.crop_w,
            ), (
                f"training target shape {tgt.shape} != "
                f"{(3, self.crop_h, self.crop_w)}"
            )

        fg = tgt[0].astype(np.float32)
        ndist = tgt[1].astype(np.float32) / 255.0

        # Critical: derive marker AFTER all geometry/resampling.
        seed = (
            (tgt[1] > SEED_U8)
            & (tgt[0] > 0)
        ).astype(np.float32)

        weight = (
            1.0
            + CFG.BORDER_WEIGHT
            * tgt[2].astype(np.float32)
        )

        x = torch.from_numpy(
            np.ascontiguousarray(img)
        ).unsqueeze(0).repeat(3, 1, 1)

        y = torch.from_numpy(
            np.ascontiguousarray(
                np.stack([fg, seed, ndist])
            )
        )

        w = torch.from_numpy(
            np.ascontiguousarray(weight)
        ).unsqueeze(0)

        return x, y, w, image_id


def normalize_batch(x):
    return (
        x - MEAN
    ) / max(
        STD,
        1e-6,
    )


# Strong sanity: deliberately force many samples so all 0/90/180/270 paths
# are likely exercised before any model is instantiated.
_ds = CellDataset(
    train_ids,
    CFG.CROP,
    True,
)

for j in range(20):
    _x, _y, _w, _id = _ds[
        random.randrange(
            len(_ds)
        )
    ]

    assert tuple(_x.shape) == (
        3,
        CFG.CROP[0],
        CFG.CROP[1],
    )

    assert tuple(_y.shape) == (
        3,
        CFG.CROP[0],
        CFG.CROP[1],
    )

    assert tuple(_w.shape) == (
        1,
        CFG.CROP[0],
        CFG.CROP[1],
    )

    if j < 3:
        print(
            "sample",
            _id,
            "|",
            tuple(_x.shape),
            "| fg%",
            round(
                float(_y[0].mean())
                * 100,
                2,
            ),
            "| seed%",
            round(
                float(_y[1].mean())
                * 100,
                3,
            ),
        )

print(
    ">>> object-centric dataset sanity passed "
    "(20 samples; fixed 400x512 under D4)"
)

del _ds, _x, _y, _w
gc.collect()


sample c17eac09ff70 | (3, 400, 512) | fg% 35.49 | seed% 5.243
sample e748ac1c469b | (3, 400, 512) | fg% 20.39 | seed% 3.344
sample 7d59ab1d21a2 | (3, 400, 512) | fg% 1.36 | seed% 0.193
>>> object-centric dataset sanity passed (20 samples; fixed 400x512 under D4)


0

In [8]:
# ── 8 · ConvNeXt-UNet with a 3-channel head, and the loss ────────────────────
class ConvBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False),
            nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False),
            nn.BatchNorm2d(cout), nn.ReLU(inplace=True))

    def forward(self, x):
        return self.block(x)


def up_to(x, ref):
    return F.interpolate(x, size=ref.shape[-2:], mode="bilinear", align_corners=False)


class UNet(nn.Module):
    def __init__(self, encoder_name, out_ch=3, dec=(256, 128, 64, 32), pretrained=True):
        super().__init__()
        self.encoder = timm.create_model(encoder_name, pretrained=pretrained,
                                         features_only=True, in_chans=3)
        chs = self.encoder.feature_info.channels()[-4:]
        self.stem = nn.Sequential(
            nn.Conv2d(3, dec[3], 3, padding=1, bias=False),
            nn.BatchNorm2d(dec[3]), nn.ReLU(inplace=True))
        self.d3 = ConvBlock(chs[3] + chs[2], dec[0])
        self.d2 = ConvBlock(dec[0] + chs[1], dec[1])
        self.d1 = ConvBlock(dec[1] + chs[0], dec[2])
        self.d0 = ConvBlock(dec[2], dec[3])
        self.dfull = ConvBlock(dec[3] * 2, dec[3])
        self.head = nn.Conv2d(dec[3], out_ch, 1)

    def forward(self, x):
        stem = self.stem(x)
        f1, f2, f3, f4 = self.encoder(x)[-4:]
        d = self.d3(torch.cat([up_to(f4, f3), f3], 1))
        d = self.d2(torch.cat([up_to(d, f2), f2], 1))
        d = self.d1(torch.cat([up_to(d, f1), f1], 1))
        d = self.d0(F.interpolate(d, scale_factor=2, mode="bilinear", align_corners=False))
        d = self.dfull(torch.cat([up_to(d, stem), stem], 1))
        return self.head(d)


def loss_fn(logits, target, weight):
    # fg + seed: weighted BCE + Dice (Stage 5 loss). dist: weighted MSE on a sigmoid.
    bin_logits, bin_target = logits[:, :2], target[:, :2]
    bce = F.binary_cross_entropy_with_logits(bin_logits, bin_target, reduction="none")
    bce = (bce * weight).mean()
    prob = torch.sigmoid(bin_logits)
    num = 2.0 * (prob * bin_target).sum(dim=(2, 3)) + 1.0
    den = prob.sum(dim=(2, 3)) + bin_target.sum(dim=(2, 3)) + 1.0
    dice = 1.0 - (num / den).mean()

    dist_pred = torch.sigmoid(logits[:, 2:3])
    dist_true = target[:, 2:3]
    fg = target[:, 0:1]
    # Distance is meaningful inside cells. Keep only a small background penalty so
    # the much larger background does not dominate the regression objective.
    dw = CFG.DIST_BG_WEIGHT + (CFG.DIST_FG_WEIGHT - CFG.DIST_BG_WEIGHT) * fg
    contact_boost = 1.0 + 0.25 * torch.clamp(weight - 1.0, min=0.0)
    dw = dw * contact_boost
    err = F.smooth_l1_loss(dist_pred, dist_true, reduction="none")
    dist_loss = (dw * err).sum() / dw.sum().clamp(min=1.0)
    return bce + dice + CFG.DIST_WEIGHT * dist_loss


@torch.no_grad()
def update_ema(model, ema_model, decay):
    for ep, p in zip(ema_model.parameters(), model.parameters()):
        ep.mul_(decay).add_(p, alpha=1.0 - decay)
    # Current BN/running-stat buffers are safer than averaging integer counters.
    for eb, b in zip(ema_model.buffers(), model.buffers()):
        eb.copy_(b)


_m = UNet(CFG.ENCODER, CFG.OUT_CHANNELS, CFG.DECODER_CHANNELS, pretrained=False).to(DEVICE).eval()
with torch.no_grad(), amp_autocast():
    _o = _m(torch.zeros(1, 3, CFG.CROP[0], CFG.CROP[1], device=DEVICE))
print("forward ok:", tuple(_o.shape))
assert _o.shape[1] == 3 and _o.shape[-2:] == CFG.CROP
del _m, _o; gc.collect(); torch.cuda.empty_cache()

forward ok: (1, 3, 400, 512)


In [9]:
# ── 9 · Inference (multi-scale + flip TTA, OOM-safe) and decoding ─────────────
def _resize_gray(img, nh, nw):
    if HAVE_CV2:
        return cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR)
    return np.asarray(Image.fromarray((img * 255).astype(np.uint8)).resize(
        (nw, nh), Image.BILINEAR), np.float32) / 255.0


def _resize_maps(p, nh, nw):
    if HAVE_CV2:
        return np.stack([cv2.resize(p[c], (nw, nh), interpolation=cv2.INTER_LINEAR)
                         for c in range(p.shape[0])])
    return np.stack([np.asarray(Image.fromarray((p[c] * 255).astype(np.uint8)).resize(
        (nw, nh), Image.BILINEAR), np.float32) / 255.0 for c in range(p.shape[0])])


def _forward_whole(model, x):
    with amp_autocast():
        return torch.sigmoid(model(x).float())


def _forward_tiled(model, x, tile=None, ov=None):
    tile = CFG.TILE if tile is None else tile
    ov = CFG.TILE_OVERLAP if ov is None else ov
    _, _, H, W = x.shape
    th, tw = min(tile, H), min(tile, W)
    th -= th % 32; tw -= tw % 32
    out = torch.zeros((1, CFG.OUT_CHANNELS, H, W), device=x.device, dtype=torch.float32)
    acc = torch.zeros((1, 1, H, W), device=x.device, dtype=torch.float32)
    ys = list(range(0, max(H - th, 0) + 1, max(1, th - ov)))
    xs = list(range(0, max(W - tw, 0) + 1, max(1, tw - ov)))
    if ys[-1] + th < H:
        ys.append(H - th)
    if xs[-1] + tw < W:
        xs.append(W - tw)
    for y0 in ys:
        for x0 in xs:
            p = _forward_whole(model, x[:, :, y0:y0 + th, x0:x0 + tw])
            out[:, :, y0:y0 + th, x0:x0 + tw] += p
            acc[:, :, y0:y0 + th, x0:x0 + tw] += 1.0
    return out / acc.clamp(min=1.0)


def _forward(model, x, force_tile=False):
    _, _, H, W = x.shape
    if force_tile or H * W > CFG.TILE_PIXEL_LIMIT:
        return _forward_tiled(model, x)
    try:
        return _forward_whole(model, x)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        print("  (OOM at", (H, W), "-> tiling)")
        return _forward_tiled(model, x)


@torch.no_grad()
def predict_maps(model, ids, image_dir, scales=None, flips=None, progress=0):
    # Returns {id: uint8 (3, IMG_H, IMG_W)} averaged over scales x flips, at native size.
    model.eval()
    scales = CFG.TTA_SCALES if scales is None else scales
    flips = CFG.TTA_FLIPS if flips is None else flips
    variants = [(False, False), (True, False), (False, True), (True, True)] if flips else [(False, False)]
    out = {}
    for j, image_id in enumerate(ids, 1):
        arr = IMAGES[image_id] if image_dir is None else load_gray_u8(image_dir / f"{image_id}.png")
        base = arr.astype(np.float32) / 255.0
        accum = np.zeros((CFG.OUT_CHANNELS, CFG.IMG_H, CFG.IMG_W), np.float32)
        n = 0
        for s in scales:
            if abs(s - 1.0) < 1e-3:
                img = base
            else:
                img = _resize_gray(base, int(round(CFG.IMG_H * s)), int(round(CFG.IMG_W * s)))
            h, w = img.shape
            ph = int(math.ceil(h / CFG.PAD_MULT) * CFG.PAD_MULT)
            pw = int(math.ceil(w / CFG.PAD_MULT) * CFG.PAD_MULT)
            x = torch.from_numpy(np.ascontiguousarray(pad_to(img, ph, pw)))
            x = normalize_batch(x.unsqueeze(0).unsqueeze(0).repeat(1, 3, 1, 1)).to(DEVICE)
            for hf, vf in variants:
                xin = x
                if hf:
                    xin = torch.flip(xin, dims=[3])
                if vf:
                    xin = torch.flip(xin, dims=[2])
                prob = _forward(model, xin)
                if vf:
                    prob = torch.flip(prob, dims=[2])
                if hf:
                    prob = torch.flip(prob, dims=[3])
                p = prob[0, :, :h, :w].float().cpu().numpy()
                if p.shape[1:] != (CFG.IMG_H, CFG.IMG_W):
                    p = _resize_maps(p, CFG.IMG_H, CFG.IMG_W)
                accum += p
                n += 1
        out[image_id] = np.clip(accum / max(n, 1) * 255.0, 0, 255).astype(np.uint8)
        if progress and j % progress == 0:
            print(f"   {j}/{len(ids)} | {fmt(elapsed())}", flush=True)
    return out


def maps_to_float(u8):
    return (u8[0].astype(np.float32) / 255.0,
            u8[1].astype(np.float32) / 255.0,
            u8[2].astype(np.float32) / 255.0)


def labels_to_flats(lab, min_area):
    # O(N log N) extraction; never loop lab == k, which dominates the sweep.
    flat = lab.reshape(-1)
    nz = np.flatnonzero(flat)
    if nz.size == 0:
        return []
    v = flat[nz]
    o = np.argsort(v, kind="stable")
    nz, v = nz[o], v[o]
    bounds = np.flatnonzero(np.diff(v)) + 1
    return [gp.astype(np.int64) for gp in np.split(nz, bounds) if gp.size >= min_area]


def watershed_labels(fg, seed, dist, marker, landscape, t_cell, t_marker):
    cell = fg >= t_cell
    if not cell.any():
        return None
    src = seed if marker == "seed" else dist
    markers, nm = ndi.label(src >= t_marker)
    if nm == 0:
        return None
    land = dist if landscape == "dist" else fg
    return watershed(-land, markers, mask=cell)


def decode(maps_u8, post):
    fg, seed, dist = maps_to_float(maps_u8)
    lab = watershed_labels(fg, seed, dist, post["marker"], post["landscape"],
                           post["t_cell"], post["t_marker"])
    if lab is None:
        return []
    return labels_to_flats(lab, post["min_area"])


@torch.no_grad()
def quick_val_f1(model, ids, post, scale):
    scores = []
    for image_id in ids:
        m = predict_maps(model, [image_id], None, scales=(scale,), flips=False)[image_id]
        scores.append(instance_f1(INST[image_id], decode(m, post)))
    return float(np.mean(scores))


print("inference + decode helpers ready")

@torch.no_grad()
def quick_val_multi(model, ids, scale):
    posts = [
        dict(marker="seed", landscape="fg", t_cell=0.50, t_marker=0.65, min_area=90),
        dict(marker="seed", landscape="dist", t_cell=0.50, t_marker=0.65, min_area=90),
        dict(marker="dist", landscape="dist", t_cell=0.50, t_marker=0.55, min_area=90),
    ]
    scores = [[] for _ in posts]
    for image_id in ids:
        maps = predict_maps(model, [image_id], None, scales=(scale,), flips=False)[image_id]
        for k, post in enumerate(posts):
            scores[k].append(instance_f1(INST[image_id], decode(maps, post)))
    means = [float(np.mean(v)) for v in scores]
    return max(means), means


inference + decode helpers ready


In [10]:
# ── 10 · PREFLIGHT — 3 training steps, one full-TTA image, projected wall clock ─
# This is the cell that protects the last training window. It must print sane numbers.
pre_model = UNet(CFG.ENCODER, CFG.OUT_CHANNELS, CFG.DECODER_CHANNELS, pretrained=True).to(DEVICE)
pre_opt = torch.optim.AdamW(pre_model.parameters(), lr=1e-4)
pre_scaler = make_scaler()
pre_loader = DataLoader(CellDataset(train_ids, CFG.CROP, True), batch_size=CFG.BATCH_SIZE,
                        shuffle=True, num_workers=0, drop_last=True)

t0 = time.time()
it = iter(pre_loader)
for _ in range(3):
    x, y, w, _ = next(it)
    x, y, w = x.to(DEVICE), y.to(DEVICE), w.to(DEVICE)
    pre_opt.zero_grad(set_to_none=True)
    with amp_autocast():
        loss = loss_fn(pre_model(normalize_batch(x)), y, w)
    pre_scaler.scale(loss).backward()
    pre_scaler.step(pre_opt); pre_scaler.update()
step_sec = (time.time() - t0) / 3
print(f"train step ~{step_sec:.2f}s (num_workers=0) | loss {float(loss):.4f}")

t0 = time.time()
m_full = predict_maps(pre_model, [val_ids[0]], None)[val_ids[0]]
tta_sec = time.time() - t0
print(f"full TTA per image ~{tta_sec:.2f}s | maps shape {m_full.shape} dtype {m_full.dtype}")

# tiling parity: the OOM fallback must agree with the whole-image path
img = _resize_gray(IMAGES[val_ids[0]].astype(np.float32) / 255.0,
                   int(round(CFG.IMG_H * 1.5)), int(round(CFG.IMG_W * 1.5)))
ph = int(math.ceil(img.shape[0] / 32) * 32); pw = int(math.ceil(img.shape[1] / 32) * 32)
xt = torch.from_numpy(np.ascontiguousarray(pad_to(img, ph, pw)))
xt = normalize_batch(xt.unsqueeze(0).unsqueeze(0).repeat(1, 3, 1, 1)).to(DEVICE)
with torch.no_grad():
    a = _forward(pre_model, xt, force_tile=False)
    b = _forward(pre_model, xt, force_tile=True)
print(f"tiling parity | mean |whole-tiled| = {float((a - b).abs().mean()):.5f} (want < 0.02)")

t0 = time.time()
_ = decode(m_full, CFG.DEFAULT_POST)
decode_sec = time.time() - t0
print(f"decode ~{decode_sec:.2f}s/image")

n_eval = max(1, (CFG.EPOCHS - CFG.EVAL_START) // CFG.EVAL_EVERY + 1)
proj_train = CFG.EPOCHS * CFG.STEPS_PER_EPOCH * step_sec * 0.75      # workers hide most of the IO
proj_eval = n_eval * CFG.QUICK_VAL_N * (tta_sec / max(len(CFG.TTA_SCALES) * 4, 1) + decode_sec)
proj_maps = (len(val_ids) + len(TEST_IDS)) * tta_sec
proj_sweep = len(CFG.T_CELL_GRID) * len(CFG.T_MARKER_GRID) * len(val_ids) * decode_sec
print("\n--- projection ---")
print(f"training  ~{fmt(proj_train)}   (deadline {fmt(CFG.TRAIN_DEADLINE_SEC)})")
print(f"in-train eval ~{fmt(proj_eval)}")
print(f"val+test TTA maps ~{fmt(proj_maps)}")
print(f"fine sweep ~{fmt(proj_sweep)}")
print(f"TOTAL ~{fmt(proj_train + proj_eval + proj_maps + proj_sweep)} "
      f"| budget {fmt(CFG.TOTAL_BUDGET_SEC)}")
if proj_train + proj_eval + proj_maps + proj_sweep > CFG.TOTAL_BUDGET_SEC:
    if len(CFG.TTA_SCALES) > 2:
        CFG.TTA_SCALES = (1.35, 1.65)
        print(">>> projected over budget: automatically reducing final TTA to", CFG.TTA_SCALES)
    else:
        print("!! projection remains tight; the hard training-time guard will protect the run.")

del pre_model, pre_opt, pre_scaler, pre_loader, it, x, y, w, xt, a, b, m_full
gc.collect(); torch.cuda.empty_cache()

train step ~14.84s (num_workers=0) | loss 1.7252
full TTA per image ~12.77s | maps shape (3, 520, 704) dtype uint8
tiling parity | mean |whole-tiled| = 0.00005 (want < 0.02)
decode ~0.00s/image

--- projection ---
training  ~9h 20m 53s   (deadline 1h 00m 00s)
in-train eval ~0h 07m 24s
val+test TTA maps ~0h 57m 03s
fine sweep ~0h 00m 12s
TOTAL ~10h 25m 33s | budget 1h 45m 00s
>>> projected over budget: automatically reducing final TTA to (1.35, 1.65)


In [11]:
# ── 11 · Train the single Stage 15 model ─────────────────────────────────────
BEST_PATH = CKPT / f"{CFG.RUN_TAG}_best.pt"
LOG_PATH = ART / f"{CFG.RUN_TAG}_train.csv"

model = UNet(CFG.ENCODER, CFG.OUT_CHANNELS, CFG.DECODER_CHANNELS, pretrained=True).to(DEVICE)
ema_model = copy.deepcopy(model).to(DEVICE).eval()
for _p in ema_model.parameters():
    _p.requires_grad_(False)
enc_params = list(model.encoder.parameters())
enc_ids = {id(p) for p in enc_params}
dec_params = [p for p in model.parameters() if id(p) not in enc_ids]
opt = torch.optim.AdamW([{"params": dec_params, "lr": CFG.LR},
                         {"params": enc_params, "lr": CFG.LR * CFG.ENCODER_LR_MULT}],
                        weight_decay=CFG.WD)
scaler = make_scaler()
loader = DataLoader(CellDataset(train_ids, CFG.CROP, True), batch_size=CFG.BATCH_SIZE,
                    shuffle=True, num_workers=CFG.NUM_WORKERS, pin_memory=True,
                    drop_last=True, persistent_workers=CFG.NUM_WORKERS > 0)

total_steps = CFG.EPOCHS * CFG.STEPS_PER_EPOCH

def lr_mult(step):
    if step < CFG.WARMUP_STEPS:
        return (step + 1) / CFG.WARMUP_STEPS
    p = (step - CFG.WARMUP_STEPS) / max(1, total_steps - CFG.WARMUP_STEPS)
    return 0.5 * (1.0 + math.cos(math.pi * min(1.0, p)))

quick_ids = QUICK_IDS
best_f1, best_epoch, history, step = -1.0, -1, [], 0
it = iter(loader)
stopped_early = False
TRAIN_T0 = time.time()

for epoch in range(1, CFG.EPOCHS + 1):
    if time.time() - TRAIN_T0 > CFG.TRAIN_DEADLINE_SEC:
        print(f"!! training deadline reached at epoch {epoch} — stopping")
        stopped_early = True
        break

    freeze = epoch <= CFG.FREEZE_ENCODER_EPOCHS
    for p in enc_params:
        p.requires_grad_(not freeze)

    model.train()
    running = 0.0
    for _ in range(CFG.STEPS_PER_EPOCH):
        try:
            x, y, w, _ = next(it)
        except StopIteration:
            it = iter(loader)
            x, y, w, _ = next(it)
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        w = w.to(DEVICE, non_blocking=True)

        mult = lr_mult(step)
        opt.param_groups[0]["lr"] = CFG.LR * mult
        opt.param_groups[1]["lr"] = 0.0 if freeze else CFG.LR * CFG.ENCODER_LR_MULT * mult

        opt.zero_grad(set_to_none=True)
        with amp_autocast():
            loss = loss_fn(model(normalize_batch(x)), y, w)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.GRAD_CLIP)
        scaler.step(opt); scaler.update()
        update_ema(model, ema_model, CFG.EMA_DECAY)
        running += float(loss.item())
        step += 1

    ep_loss = running / CFG.STEPS_PER_EPOCH
    msg = f"ep {epoch:02d}/{CFG.EPOCHS} loss {ep_loss:.4f}"

    due = epoch >= CFG.EVAL_START and ((epoch - CFG.EVAL_START) % CFG.EVAL_EVERY == 0
                                       or epoch == CFG.EPOCHS)
    if due:
        q, qfam = quick_val_multi(ema_model, quick_ids, CFG.QUICK_SCALE)
        history.append(dict(epoch=epoch, loss=ep_loss, f1=q,
                            seed_fg=qfam[0], seed_dist=qfam[1], dist_dist=qfam[2]))
        msg += (f" | quick {q:.4f} "
                f"[seed/fg {qfam[0]:.4f}, seed/dist {qfam[1]:.4f}, dist/dist {qfam[2]:.4f}]")
        if q > best_f1:
            best_f1, best_epoch = q, epoch
            torch.save({"state_dict": ema_model.state_dict(), "encoder": CFG.ENCODER,
                        "run_tag": CFG.RUN_TAG, "epoch": epoch, "quick_f1": q,
                        "quick_family_scores": qfam, "ema_decay": CFG.EMA_DECAY}, BEST_PATH)
            msg += "  *"
    print(msg, "|", fmt(elapsed()), flush=True)

if best_f1 < 0:   # deadline hit before the first eval — keep the last weights anyway
    torch.save({"state_dict": ema_model.state_dict(), "encoder": CFG.ENCODER,
                "run_tag": CFG.RUN_TAG, "epoch": epoch, "quick_f1": -1.0,
                "quick_family_scores": None, "ema_decay": CFG.EMA_DECAY}, BEST_PATH)
    print("!! no evaluated checkpoint — saved final weights instead")

pd.DataFrame(history).to_csv(LOG_PATH, index=False)
STATS.update(best_epoch=best_epoch, best_quick_f1=best_f1, stopped_early=stopped_early,
             train_seconds=time.time() - TRAIN_T0, object_centric_p=CFG.OBJECT_CENTRIC_P,
             ema_decay=CFG.EMA_DECAY)
print(f"\nBEST epoch {best_epoch} | quick F1 {best_f1:.4f} | {BEST_PATH}")
if DRIVE_DIR is not None:
    try:
        shutil.copy2(BEST_PATH, DRIVE_DIR / BEST_PATH.name)
        print("checkpoint mirrored to Drive")
    except Exception as exc:
        print("Drive copy failed:", exc)

del model, ema_model, opt, scaler, loader, it
gc.collect(); torch.cuda.empty_cache()

ep 01/72 loss 1.7376 | 0h 03m 27s
ep 02/72 loss 1.5739 | 0h 03m 37s
ep 03/72 loss 1.4642 | 0h 03m 49s
ep 04/72 loss 1.3791 | 0h 04m 02s
ep 05/72 loss 1.3106 | 0h 04m 15s
ep 06/72 loss 1.2408 | 0h 04m 29s
ep 07/72 loss 1.1765 | 0h 04m 42s
ep 08/72 loss 1.1338 | 0h 04m 56s
ep 09/72 loss 1.0858 | 0h 05m 09s
ep 10/72 loss 1.0456 | 0h 05m 23s
ep 11/72 loss 0.9896 | 0h 05m 37s
ep 12/72 loss 0.9558 | 0h 05m 51s
ep 13/72 loss 0.9194 | 0h 06m 05s
ep 14/72 loss 0.9029 | 0h 06m 19s
ep 15/72 loss 0.8588 | 0h 06m 33s
ep 16/72 loss 0.8468 | 0h 06m 47s
ep 17/72 loss 0.8251 | 0h 07m 01s
ep 18/72 loss 0.7971 | 0h 07m 15s
ep 19/72 loss 0.7831 | 0h 07m 29s
ep 20/72 loss 0.7617 | 0h 07m 43s
ep 21/72 loss 0.7354 | 0h 07m 57s
ep 22/72 loss 0.7268 | 0h 08m 11s
ep 23/72 loss 0.7174 | 0h 08m 26s
ep 24/72 loss 0.6990 | quick 0.1125 [seed/fg 0.0000, seed/dist 0.0000, dist/dist 0.1125]  * | 0h 08m 43s
ep 25/72 loss 0.6964 | 0h 08m 58s
ep 26/72 loss 0.6723 | 0h 09m 13s
ep 27/72 loss 0.6668 | 0h 09m 27s
ep 28/72 lo

In [12]:
# ── 12 · Reload best, cache val + test TTA probability maps ──────────────────
try:
    blob = torch.load(BEST_PATH, map_location="cpu", weights_only=False)
except TypeError:
    blob = torch.load(BEST_PATH, map_location="cpu")
assert blob["run_tag"] == CFG.RUN_TAG
best_model = UNet(CFG.ENCODER, CFG.OUT_CHANNELS, CFG.DECODER_CHANNELS, pretrained=False).to(DEVICE)
best_model.load_state_dict(blob["state_dict"])
best_model.eval()
print("loaded checkpoint from epoch", blob["epoch"], "| quick F1", round(float(blob["quick_f1"]), 4))

VAL_PATH, TEST_PATH = MAPS / f"{CFG.RUN_TAG}_val.npz", MAPS / f"{CFG.RUN_TAG}_test.npz"
if CFG.RESUME and VAL_PATH.exists() and TEST_PATH.exists():
    with np.load(VAL_PATH) as z:
        VAL_MAPS = {k: z[k] for k in z.files}
    with np.load(TEST_PATH) as z:
        TEST_MAPS = {k: z[k] for k in z.files}
    print("maps loaded from cache")
else:
    t0 = time.time()
    VAL_MAPS = predict_maps(best_model, val_ids, None, progress=20)
    print("val maps", fmt(time.time() - t0))
    t0 = time.time()
    TEST_MAPS = predict_maps(best_model, TEST_IDS, TEST_DIR, progress=40)
    print("test maps", fmt(time.time() - t0))
    np.savez_compressed(VAL_PATH, **VAL_MAPS)
    np.savez_compressed(TEST_PATH, **TEST_MAPS)
    print("maps cached:", VAL_PATH.name, TEST_PATH.name)

assert set(VAL_MAPS) == set(val_ids) and set(TEST_MAPS) == set(TEST_IDS)
print("elapsed:", fmt(elapsed()))

# Mirror the expensive probability maps immediately.
if DRIVE_DIR is not None:
    for _p in (VAL_PATH, TEST_PATH):
        try:
            shutil.copy2(_p, DRIVE_DIR / _p.name)
        except Exception as exc:
            print("Drive map copy failed for", _p.name, exc)


loaded checkpoint from epoch 72 | quick F1 0.7984
   20/86 | 0h 21m 20s
   40/86 | 0h 21m 31s
   60/86 | 0h 21m 42s
   80/86 | 0h 21m 54s
val maps 0h 00m 56s
   40/182 | 0h 22m 20s
   80/182 | 0h 22m 42s
   120/182 | 0h 23m 05s
   160/182 | 0h 23m 28s
test maps 0h 01m 43s
maps cached: stage15_final_objectcentric_zoomdist_val.npz stage15_final_objectcentric_zoomdist_test.npz
elapsed: 0h 23m 51s


In [13]:
# ── 13 · COARSE probe: which decode family? (markers x landscape) ────────────
# One trained model, four decodes. If the distance head is weak, ("seed","fg") is exactly
# the Stage 10 decode and the run degrades gracefully instead of failing.
probe_ids = val_ids[:CFG.COARSE_N]
ref_marker = {"seed": 0.65, "dist": 0.55}
family_rows = []
for marker, landscape in CFG.COARSE_FAMILIES:
    post = dict(marker=marker, landscape=landscape, t_cell=0.50,
                t_marker=ref_marker[marker], min_area=90)
    scores = [instance_f1(INST[i], decode(VAL_MAPS[i], post)) for i in probe_ids]
    family_rows.append(dict(marker=marker, landscape=landscape,
                            t_marker=post["t_marker"], f1=float(np.mean(scores))))
    print(f"markers={marker:4s} landscape={landscape:4s} -> F1 {np.mean(scores):.4f}")

FAMILY = pd.DataFrame(family_rows).sort_values("f1", ascending=False).reset_index(drop=True)
BEST_MARKER = FAMILY.iloc[0]["marker"]
BEST_LANDSCAPE = FAMILY.iloc[0]["landscape"]
print(f"\nchosen family: markers={BEST_MARKER}, landscape={BEST_LANDSCAPE}")
print("(ranked on", len(probe_ids), "images at one reference setting — the fine sweep decides thresholds)")
STATS["decode_family"] = dict(marker=BEST_MARKER, landscape=BEST_LANDSCAPE,
                              table=FAMILY.to_dict("records"))

markers=seed landscape=dist -> F1 0.8079
markers=seed landscape=fg   -> F1 0.8185
markers=dist landscape=dist -> F1 0.8015
markers=dist landscape=fg   -> F1 0.8170

chosen family: markers=seed, landscape=fg
(ranked on 40 images at one reference setting — the fine sweep decides thresholds)


In [14]:
# ── 14 · FINE sweep on all 86 validation images, then the plateau centre ─────
PER = {}          # (t_cell, t_marker, min_area) -> per-image F1 array
CNT = {}          # same key -> per-image predicted count
rows = []
t0 = time.time()
for tc in CFG.T_CELL_GRID:
    for tm in CFG.T_MARKER_GRID:
        decoded = {}
        for image_id in val_ids:
            fg, seed, dist = maps_to_float(VAL_MAPS[image_id])
            lab = watershed_labels(fg, seed, dist, BEST_MARKER, BEST_LANDSCAPE, tc, tm)
            decoded[image_id] = None if lab is None else lab
        for ma in CFG.MIN_AREA_GRID:
            f1s, counts = [], []
            for image_id in val_ids:
                lab = decoded[image_id]
                pred = [] if lab is None else labels_to_flats(lab, ma)
                f1s.append(instance_f1(INST[image_id], pred))
                counts.append(len(pred))
            key = (tc, tm, ma)
            PER[key] = np.asarray(f1s, np.float64)
            CNT[key] = np.asarray(counts, np.int64)
            rows.append(dict(t_cell=tc, t_marker=tm, min_area=ma,
                             f1=float(np.mean(f1s)), mean_pred=float(np.mean(counts))))
        del decoded
    print(f"  t_cell {tc} done | {fmt(elapsed())}", flush=True)
print("sweep took", fmt(time.time() - t0))

SWEEP = pd.DataFrame(rows).sort_values("f1", ascending=False).reset_index(drop=True)
print(SWEEP.head(12).to_string(index=False))

peak = float(SWEEP.iloc[0].f1)
plateau = SWEEP[SWEEP.f1 >= peak - CFG.PLATEAU_TOL]

def snap(v, grid):
    return min(grid, key=lambda x: abs(x - v))

POST = dict(marker=BEST_MARKER, landscape=BEST_LANDSCAPE,
            t_cell=snap(float(np.median(plateau.t_cell)), CFG.T_CELL_GRID),
            t_marker=snap(float(np.median(plateau.t_marker)), CFG.T_MARKER_GRID),
            min_area=int(snap(float(np.median(plateau.min_area)), CFG.MIN_AREA_GRID)))
KEY = (POST["t_cell"], POST["t_marker"], POST["min_area"])
VAL_PER_IMAGE = PER[KEY]
VAL_F1 = float(VAL_PER_IMAGE.mean())

print(f"\npeak {peak:.4f} | plateau size {len(plateau)} | chosen {POST}")
print(f"SELECTED LOCAL F1 = {VAL_F1:.4f}")
print(f"  Stage 10 reference {CFG.REF_STAGE10_LOCAL:.4f} (public 0.5295)")
print(f"  Stage 12 reference {CFG.REF_STAGE12_LOCAL:.4f} (public 0.5335)  <- the bar")
print(f"  delta vs Stage 12: {VAL_F1 - CFG.REF_STAGE12_LOCAL:+.4f}")
STATS.update(sweep_peak=peak, val_f1=VAL_F1, post={k: str(v) for k, v in POST.items()})
SWEEP.to_csv(ART / f"{CFG.RUN_TAG}_sweep.csv", index=False)

  t_cell 0.4 done | 0h 24m 18s
  t_cell 0.45 done | 0h 24m 39s
  t_cell 0.5 done | 0h 25m 01s
  t_cell 0.55 done | 0h 25m 21s
  t_cell 0.6 done | 0h 25m 42s
sweep took 0h 01m 46s
 t_cell  t_marker  min_area       f1  mean_pred
   0.55      0.65        90 0.814451 109.348837
   0.50      0.65        90 0.814391 110.127907
   0.45      0.65        90 0.814316 110.732558
   0.55      0.60        90 0.813557 109.104651
   0.40      0.65       110 0.813383 108.011628
   0.50      0.60        90 0.813337 109.941860
   0.60      0.65        90 0.813296 108.558140
   0.45      0.60        90 0.813205 110.546512
   0.45      0.70        90 0.813126 110.825581
   0.50      0.70        90 0.813079 110.197674
   0.45      0.65       110 0.813061 107.209302
   0.50      0.65        70 0.812995 112.895349

peak 0.8145 | plateau size 56 | chosen {'marker': 'seed', 'landscape': 'fg', 't_cell': 0.5, 't_marker': 0.65, 'min_area': 90}
SELECTED LOCAL F1 = 0.8144
  Stage 10 reference 0.8072 (public 0.5295)

In [15]:
# ── 15 · Density-conditioned t_cell, with a bootstrap gate ───────────────────
# Free: every combination is already in PER. Stage 9 measured +0.0049 for this idea and
# rejected it on a +0.005 gate. Selecting per-bin on the same 86 images IS selection bias,
# so the gate here is deliberately strict.
def bin_of(count):
    for b in range(len(CFG.DENSITY_LABELS)):
        if CFG.DENSITY_EDGES[b] <= count < CFG.DENSITY_EDGES[b + 1]:
            return b
    return len(CFG.DENSITY_LABELS) - 1

val_bins = np.array([bin_of(c) for c in CNT[KEY]])
route_tcell, route_rows = {}, []
routed = VAL_PER_IMAGE.copy()
for b, label in enumerate(CFG.DENSITY_LABELS):
    m = val_bins == b
    if m.sum() == 0:
        route_tcell[b] = POST["t_cell"]
        continue
    best_tc, best_v = POST["t_cell"], float(VAL_PER_IMAGE[m].mean())
    base_v = best_v
    for tc in CFG.T_CELL_GRID:
        v = float(PER[(tc, POST["t_marker"], POST["min_area"])][m].mean())
        if v > best_v + 1e-9:
            best_tc, best_v = tc, v
    route_tcell[b] = best_tc
    routed[m] = PER[(best_tc, POST["t_marker"], POST["min_area"])][m]
    route_rows.append(dict(bin=label, images=int(m.sum()), t_cell=best_tc,
                           base=round(base_v, 4), routed=round(best_v, 4)))

ROUTE_TABLE = pd.DataFrame(route_rows)
print(ROUTE_TABLE.to_string(index=False) if len(ROUTE_TABLE) else "(no bins)")
routed_f1 = float(routed.mean())
gain = routed_f1 - VAL_F1

rs = np.random.RandomState(0)
idx = rs.randint(0, len(val_ids), size=(2000, len(val_ids)))
wins = float(np.mean(routed[idx].mean(1) > VAL_PER_IMAGE[idx].mean(1)))
print(f"\nglobal {VAL_F1:.4f} | routed {routed_f1:.4f} | gain {gain:+.4f} | bootstrap win-rate {wins:.2f}")

USE_ROUTE = bool(gain >= CFG.ROUTE_MIN_GAIN and wins >= CFG.ROUTE_MIN_WINRATE)
print("DECISION:", "USE density-conditioned t_cell" if USE_ROUTE else
      "KEEP the single global decode (gate not cleared)")
FINAL_PER_IMAGE = routed if USE_ROUTE else VAL_PER_IMAGE
FINAL_F1 = float(FINAL_PER_IMAGE.mean())
STATS.update(routed_f1=routed_f1, route_gain=gain, route_winrate=wins,
             use_route=USE_ROUTE, final_f1=FINAL_F1,
             route_tcell={CFG.DENSITY_LABELS[b]: v for b, v in route_tcell.items()})
print(f"FINAL LOCAL F1 = {FINAL_F1:.4f}")

             bin  images  t_cell   base  routed
      sparse <50      45    0.45 0.8514  0.8527
   medium 50-199      25    0.50 0.7544  0.7544
   dense 200-399      12    0.60 0.7963  0.7991
very dense >=400       4    0.50 0.8272  0.8272

global 0.8144 | routed 0.8155 | gain +0.0011 | bootstrap win-rate 0.81
DECISION: KEEP the single global decode (gate not cleared)
FINAL LOCAL F1 = 0.8144


In [16]:
# ── 16 · Diagnostics: density audit + near-miss census (for the write-up) ─────
audit = []
for image_id, f1 in zip(val_ids, FINAL_PER_IMAGE):
    post_i = dict(POST)
    if USE_ROUTE:
        post_i["t_cell"] = route_tcell[bin_of(len(decode(VAL_MAPS[image_id], POST)))]
    pred = decode(VAL_MAPS[image_id], post_i)
    ng, npred = len(INST[image_id]), len(pred)
    tp = f1 * (ng + npred) / 2.0
    audit.append(dict(id=image_id, n_gt=ng, n_pred=npred, f1=float(f1),
                      fn=ng - tp, fp=npred - tp))

AUDIT = pd.DataFrame(audit)
AUDIT["bin"] = pd.cut(AUDIT.n_gt, list(CFG.DENSITY_EDGES),
                      labels=list(CFG.DENSITY_LABELS), right=False)
DENSITY = AUDIT.groupby("bin", observed=False).agg(
    images=("id", "size"), mean_f1=("f1", "mean"), mean_gt=("n_gt", "mean"),
    mean_pred=("n_pred", "mean"), mean_fn=("fn", "mean"), mean_fp=("fp", "mean"))
DENSITY["pred_over_gt"] = DENSITY.mean_pred / DENSITY.mean_gt.clip(lower=1e-9)
print(DENSITY.round(4).to_string())
print("\nStage 10 e70 reference: sparse 0.8573 / medium 0.7569 / dense 0.7792 / very dense 0.7518")

# near-miss census: where does each GT cell land on the Dice axis?
buckets = np.zeros(4, np.int64)   # hit >=0.50 | near 0.35-0.50 | weak 0.10-0.35 | missed <0.10
by_bin = defaultdict(lambda: np.zeros(4, np.int64))
for image_id in val_ids[:60]:
    pred = decode(VAL_MAPS[image_id], POST)
    gt = INST[image_id]
    if not gt:
        continue
    d = dice_matrix(gt, pred)
    best = d.max(axis=1) if d.size else np.zeros(len(gt), np.float32)
    b = bin_of(len(gt))
    for v in best:
        k = 0 if v >= 0.50 else 1 if v >= 0.35 else 2 if v >= 0.10 else 3
        buckets[k] += 1
        by_bin[b][k] += 1

tot = max(int(buckets.sum()), 1)
NEARMISS = {"hit>=0.50": round(100 * buckets[0] / tot, 1),
            "near 0.35-0.50": round(100 * buckets[1] / tot, 1),
            "weak 0.10-0.35": round(100 * buckets[2] / tot, 1),
            "missed <0.10": round(100 * buckets[3] / tot, 1)}
print("\nnear-miss census (60 val images):", NEARMISS)
print("Stage 12 reference: hit 77.0 | near 8.6 | weak 8.3 | missed 6.1")
for b in sorted(by_bin):
    arr = by_bin[b]
    s = max(int(arr.sum()), 1)
    print(f"  {CFG.DENSITY_LABELS[b]:18s} hit {100*arr[0]/s:5.1f}%  near {100*arr[1]/s:5.1f}%"
          f"  weak {100*arr[2]/s:5.1f}%  missed {100*arr[3]/s:5.1f}%")
STATS["near_miss"] = NEARMISS

                  images  mean_f1   mean_gt  mean_pred   mean_fn  mean_fp  pred_over_gt
bin                                                                                    
sparse <50            41   0.8567   27.3902    27.6585    3.9512   4.2195        1.0098
medium 50-199         28   0.7699   91.8929    94.8214   20.4286  23.3571        1.0319
dense 200-399         11   0.7917  320.6364   293.3636   75.9091  48.6364        0.9149
very dense >=400       6   0.7749  476.8333   409.1667  128.3333  60.6667        0.8581

Stage 10 e70 reference: sparse 0.8573 / medium 0.7569 / dense 0.7792 / very dense 0.7518

near-miss census (60 val images): {'hit>=0.50': np.float64(76.5), 'near 0.35-0.50': np.float64(9.0), 'weak 0.10-0.35': np.float64(9.1), 'missed <0.10': np.float64(5.3)}
Stage 12 reference: hit 77.0 | near 8.6 | weak 8.3 | missed 6.1
  sparse <50         hit  87.3%  near   3.5%  weak   4.1%  missed   5.2%
  medium 50-199      hit  78.7%  near   9.7%  weak   6.2%  missed   5.3%
  

In [17]:
# ── 17 · RLE parity, then write 15th_submission.csv ──────────────────────────
parity_direct, parity_rt = [], []
for j, image_id in enumerate(val_ids[:25]):
    post_i = dict(POST)
    if USE_ROUTE:
        post_i["t_cell"] = route_tcell[bin_of(int(CNT[KEY][j]))]
    pred = decode(VAL_MAPS[image_id], post_i)
    rt = [np.unique(rle_decode_flat(flat_to_rle(f), CFG.IMG_H, CFG.IMG_W, RLE_ORDER))
          for f in pred]
    parity_direct.append(FINAL_PER_IMAGE[j])
    parity_rt.append(instance_f1(INST[image_id], rt))
delta = abs(float(np.mean(parity_direct)) - float(np.mean(parity_rt)))
print(f"parity | in-memory {np.mean(parity_direct):.4f} | after RLE {np.mean(parity_rt):.4f} "
      f"| delta {delta:.5f}")
assert delta < 0.002, "RLE encoding is losing score — do not submit."
STATS["parity_delta"] = float(delta)

SUB_PATH = Path("/content/15th_submission.csv")
rows, counts, fallbacks = [], [], 0
for image_id in TEST_IDS:
    post_i = dict(POST)
    if USE_ROUTE:
        base_pred = decode(TEST_MAPS[image_id], POST)
        post_i["t_cell"] = route_tcell[bin_of(len(base_pred))]
        pred = base_pred if post_i["t_cell"] == POST["t_cell"] else decode(TEST_MAPS[image_id], post_i)
    else:
        pred = decode(TEST_MAPS[image_id], POST)
    if not pred:      # an image with GT and zero masks scores exactly 0.0 — never allow it
        fallbacks += 1
        pred = decode(TEST_MAPS[image_id],
                      dict(POST, t_cell=0.30, t_marker=0.30, min_area=1))[:1]
    if not pred:
        y0 = CFG.IMG_H // 2
        pred = [np.arange(y0 * CFG.IMG_W, y0 * CFG.IMG_W + 25, dtype=np.int64)]
    counts.append(len(pred))
    rows.extend((image_id, flat_to_rle(f)) for f in pred)

sub = pd.DataFrame(rows, columns=[ID_COL, ANN_COL])
sub.insert(0, CFG.SUB_INDEX_COL, np.arange(len(sub), dtype=np.int64))
sub.to_csv(SUB_PATH, index=False)
assert set(sub[ID_COL].astype(str)) == set(TEST_IDS), "Submission is missing test images."
assert sub[ANN_COL].notna().all() and sub[CFG.SUB_INDEX_COL].is_unique
assert (sub[ANN_COL].astype(str).str.len() > 0).all()

SUB_INFO = dict(rows=int(len(sub)), mean=float(np.mean(counts)),
                min=int(np.min(counts)), max=int(np.max(counts)), fallbacks=int(fallbacks))
print("wrote:", SUB_PATH, "|", SUB_INFO)
print(f"predictions/image {SUB_INFO['mean']:.1f} vs GT train mean "
      f"{STATS['gt_mean_instances']:.1f} | Stage 12 was 106.9")
STATS["submission"] = SUB_INFO

parity | in-memory 0.8331 | after RLE 0.8331 | delta 0.00000
wrote: /content/15th_submission.csv | {'rows': 19845, 'mean': 109.03846153846153, 'min': 15, 'max': 592, 'fallbacks': 0}
predictions/image 109.0 vs GT train mean 125.2 | Stage 12 was 106.9


In [18]:
# ── 18 · Report, artifact preservation, downloads, and the decision rule ─────
REPORT_PATH = Path("/content/15th_REPORT.md")
SELECTED_MODEL_PATH = Path("/content/stage15_selected_model.pt")
shutil.copy2(BEST_PATH, SELECTED_MODEL_PATH)
dense_text = "\n".join("    " + s for s in DENSITY.round(4).to_string().splitlines())
fam_text = "\n".join("    " + s for s in FAMILY.round(4).to_string(index=False).splitlines())
route_text = ("\n".join("    " + s for s in ROUTE_TABLE.to_string(index=False).splitlines())
              if len(ROUTE_TABLE) else "    (no routing table)")

verdict = ("SUBMIT — clears the Stage 12 bar" if FINAL_F1 >= CFG.REF_STAGE12_LOCAL + 0.004 else
           "SUBMIT AS A DIVERSITY SLOT — comparable to Stage 12, different failure modes"
           if FINAL_F1 >= CFG.REF_STAGE12_LOCAL - 0.004 else
           "DO NOT SELECT — below Stage 12 locally; keep 12th + 5th")

lines = [
    "# Stage 15 FINAL report — object-centric zoom-in + continuous distance field + distance-watershed",
    "",
    "## Formulation",
    f"- encoder: {CFG.ENCODER} (ImageNet-22k -> 1k), U-Net decoder {CFG.DECODER_CHANNELS}",
    "- head: 3 channels — foreground, e70 seed, normalised per-instance distance field",
    f"- object-centric training probability: {CFG.OBJECT_CENTRIC_P:.0%}",
    f"- global training instance pool: {len(TRAIN_INSTANCE_POOL)} cells",
    f"- centre jitter: {CFG.CENTER_JITTER_FRAC:.2f} of native context size",
    f"- EMA decay: {CFG.EMA_DECAY}",
    f"- targets: seed DERIVED by thresholding the resized distance field at {CFG.SEED_EROSION_FRAC}",
    f"- augmentation: ZOOM-IN mixture {CFG.SCALE_MIX}, D4={CFG.AUG_D4}, flips, "
    f"brightness/contrast/gamma/blur/noise",
    f"- crop {CFG.CROP}, batch {CFG.BATCH_SIZE}, {CFG.EPOCHS} epochs x {CFG.STEPS_PER_EPOCH} steps",
    f"- inference: scales {CFG.TTA_SCALES} x 4 flips, maps averaged at native resolution",
    "",
    "## Validation",
    f"- grouped split fold {CFG.FOLD}/{CFG.N_FOLDS} | groups {STATS['n_groups']} | "
    f"train/val {STATS['n_train']}/{STATS['n_val']} | max val->train corr {STATS['max_val_train_corr']:.3f}",
    f"- checkpoint epoch {blob['epoch']} | quick F1 {float(blob['quick_f1']):.4f}",
    f"- decode family probe:",
    fam_text,
    f"- chosen decode: {POST}",
    f"- sweep peak {peak:.4f} | plateau-centre F1 {VAL_F1:.4f}",
    f"- density-conditioned t_cell: gain {STATS['route_gain']:+.4f}, bootstrap win-rate "
    f"{STATS['route_winrate']:.2f}, applied = {USE_ROUTE}",
    route_text,
    f"- **FINAL LOCAL F1 = {FINAL_F1:.4f}**",
    f"- references: Stage 10 {CFG.REF_STAGE10_LOCAL:.4f} (public 0.5295), "
    f"Stage 12 {CFG.REF_STAGE12_LOCAL:.4f} (public 0.5335)",
    "",
    "## Density validation",
    dense_text,
    "",
    "## Near-miss census (share of GT cells by best achievable Dice)",
    f"- Stage 15: {STATS['near_miss']}",
    "- Stage 12 reference: hit 77.0 | near 8.6 | weak 8.3 | missed 6.1",
    "",
    "## Submission integrity",
    f"- RLE order {RLE_ORDER} | parity delta {delta:.6f}",
    f"- predictions/image mean {SUB_INFO['mean']:.1f} (min {SUB_INFO['min']}, max {SUB_INFO['max']})",
    f"- fallbacks {SUB_INFO['fallbacks']} | all {len(TEST_IDS)} test IDs present",
    f"- runtime {fmt(elapsed())}",
    "",
    "## Verdict",
    f"- {verdict}",
    "",
    "## Outputs",
    "- /content/15th_submission.csv",
    "- /content/15th_REPORT.md",
    f"- checkpoint {BEST_PATH}",
    f"- convenience model copy {SELECTED_MODEL_PATH}",
    f"- maps {MAPS}",
]
report = "\n".join(lines)
REPORT_PATH.write_text(report)
(ART / "15th_REPORT.md").write_text(report)
shutil.copy2(SUB_PATH, ART / "15th_submission.csv")
(ART / "stats.json").write_text(json.dumps(STATS, indent=1, default=str))
if DRIVE_DIR is not None:
    for p in (SUB_PATH, REPORT_PATH, SELECTED_MODEL_PATH, ART / "stats.json",
              ART / f"{CFG.RUN_TAG}_sweep.csv", MAPS / f"{CFG.RUN_TAG}_val.npz",
              MAPS / f"{CFG.RUN_TAG}_test.npz"):
        try:
            shutil.copy2(p, DRIVE_DIR / p.name)
        except Exception as exc:
            print("Drive copy failed for", p.name, exc)

print(report)
print("\nKaggle description to paste:")
print(f"Y_15th FINAL: single ConvNeXt-Tiny U-Net, 70% object-centric cell crop/upscale training with a continuous "
      f"normalised distance target, seeds derived post-resize at 0.70, {POST['marker']}/{POST['landscape']} watershed, "
      f"{len(CFG.TTA_SCALES)}-scale x 4-flip TTA, grouped-val F1 {FINAL_F1:.4f}, no ensemble.")

try:
    from google.colab import files
    files.download(str(SUB_PATH))
    files.download(str(REPORT_PATH))
except Exception as exc:
    print("auto-download skipped:", exc)

# Stage 15 FINAL report — object-centric zoom-in + continuous distance field + distance-watershed

## Formulation
- encoder: convnext_tiny.fb_in22k_ft_in1k (ImageNet-22k -> 1k), U-Net decoder (256, 128, 64, 32)
- head: 3 channels — foreground, e70 seed, normalised per-instance distance field
- object-centric training probability: 70%
- global training instance pool: 42981 cells
- centre jitter: 0.18 of native context size
- EMA decay: 0.9985
- targets: seed DERIVED by thresholding the resized distance field at 0.7
- augmentation: ZOOM-IN mixture ((0.25, 1.15, 1.35), (0.5, 1.35, 1.65), (0.25, 1.65, 1.9)), D4=True, flips, brightness/contrast/gamma/blur/noise
- crop (400, 512), batch 8, 72 epochs x 42 steps
- inference: scales (1.35, 1.65) x 4 flips, maps averaged at native resolution

## Validation
- grouped split fold 0/5 | groups 421 | train/val 338/86 | max val->train corr 0.665
- checkpoint epoch 72 | quick F1 0.7984
- decode family probe:
    marker landscape  t_marker     f1
      

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Final submission rule

This is the **last Kaggle submission available**. Once Cell 17 passes RLE parity and submission-integrity checks, upload `15th_submission.csv`; leaving the last submission unused has no upside.

For the two final selected slots:

- **Stage 12 (0.5335) stays selected.**
- If Stage 15 publicly beats or meaningfully complements the old Stage 5 diversity slot, select **12 + 15**.
- If Stage 15 is clearly weak publicly, keep **12 + 5**.

Stage 15 is deliberately a different single-model formulation: object-centric crop/upscale training, continuous distance supervision, EMA, and distance-aware post-processing rather than another model ensemble.
